In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [3]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def get_robust_session():
    """Creates a session that automatically retries when the server drops the connection."""
    session = requests.Session()
    retry = Retry(
        total=5,  # Retry 5 times
        backoff_factor=1,  # Wait 1s, 2s, 4s, 8s, 16s between retries
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def scrape_goobjoog_links(base_url, output_file="goobjoog_ciyaaraha_links.xlsx"):
    all_links = set()
    page = 1
    
    # More realistic headers to bypass basic bot detection
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.google.com/'
    }

    session = get_robust_session()

    # Resume from existing file if it exists
    if os.path.exists(output_file):
        try:
            existing_df = pd.read_excel(output_file)
            all_links = set(existing_df['URL'].tolist())
            print(f"📂 Loaded {len(all_links)} existing links from file.")
        except:
            pass

    print(f"🚀 Starting scrape on: {base_url}")

    while True:
        current_url = f"{base_url}page/{page}/" if page > 1 else base_url
        print(f"🌐 Fetching Page {page}: {current_url}")
        
        try:
            # Use the session instead of requests.get
            response = session.get(current_url, headers=headers, timeout=25)
            
            if response.status_code == 404:
                print(f"🏁 Reached the end (404).")
                break
            elif response.status_code != 200:
                print(f"⚠️ Status {response.status_code} on page {page}. Retrying next...")
                page += 1
                continue

            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Goobjoog's current structure often uses h3 tags for headlines in category pages
            # We look for all <a> tags within the main content area
            links = soup.find_all('a', rel='bookmark')
            
            if not links:
                # If 'bookmark' isn't working, try a broader search for links in the main container
                main_content = soup.find('main') or soup.find('div', id='primary')
                if main_content:
                    links = main_content.find_all('a', href=True)

            if not links:
                print("⚠️ No links found. The site layout might have changed.")
                break

            new_links_count = 0
            for a in links:
                href = a.get('href')
                # Filter to ensure we only get actual articles (usually contain a date or specific path)
                if href and "/20" in href and href not in all_links: 
                    all_links.add(href)
                    new_links_count += 1

            print(f"   ✅ Found {new_links_count} new links. (Total Unique: {len(all_links)})")

            # Checkpoint save
            if page % 3 == 0:
                pd.DataFrame(list(all_links), columns=["URL"]).to_excel(output_file, index=False)
                print(f"   💾 Checkpoint saved.")

            page += 1
            time.sleep(3) # Increased delay to be safer

        except Exception as e:
            print(f"❌ Connection error: {e}")
            print("🕒 Waiting 10 seconds before trying again...")
            time.sleep(10)
            continue # Try the next page instead of breaking

    # Final Save
    if all_links:
        df = pd.DataFrame(list(all_links), columns=["URL"])
        df.to_excel(output_file, index=False)
        print(f"\n✅ SUCCESS! Total unique links: {len(all_links)}")

if __name__ == "__main__":
    target_url = "https://goobjoog.com/qayb/cayaaraha/"
    scrape_goobjoog_links(target_url)

📂 Loaded 100 existing links from file.
🚀 Starting scrape on: https://goobjoog.com/qayb/cayaaraha/
🌐 Fetching Page 1: https://goobjoog.com/qayb/cayaaraha/
❌ Connection error: HTTPSConnectionPool(host='goobjoog.com', port=443): Max retries exceeded with url: /qayb/cayaaraha/ (Caused by ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)))
🕒 Waiting 10 seconds before trying again...
🌐 Fetching Page 1: https://goobjoog.com/qayb/cayaaraha/
   ✅ Found 0 new links. (Total Unique: 100)
🌐 Fetching Page 2: https://goobjoog.com/qayb/cayaaraha/page/2/
   ✅ Found 0 new links. (Total Unique: 100)
🌐 Fetching Page 3: https://goobjoog.com/qayb/cayaaraha/page/3/
   ✅ Found 0 new links. (Total Unique: 100)
   💾 Checkpoint saved.
🌐 Fetching Page 4: https://goobjoog.com/qayb/cayaaraha/page/4/
   ✅ Found 0 new links. (Total Unique: 100)
🌐 Fetching Page 5: https://goobjoog.com/qayb/cayaaraha/page/5/
   ✅ Found 0 

In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def scrape_somali_news(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        # Added a 10-second timeout to prevent the script from hanging on slow links
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Get the Headline
            headline = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Headline Found"
            
            # Get the Body
            body_content = ""
            article_div = soup.find('div', class_='entry-content') or \
                          soup.find('div', class_='article-content') or \
                          soup.find('div', id='content-main')
            
            if article_div:
                paragraphs = article_div.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs])
            else:
                # Fallback to all paragraphs if specific container isn't found
                paragraphs = soup.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs])

            return headline, body_content
        else:
            return None, f"Error: Status {response.status_code}"
    except Exception as e:
        return None, f"Request failed: {str(e)}"

# 1. Load the original file
input_file = "goobjoog_caafimaad_articles.xlsx"
df = pd.read_excel(input_file)

# 2. Define the target rows
# Excel rows 230 to 642 map to zero-based indices 229 to 641
start_index = 229 
end_index = 642 

print(f"🔄 Starting repair process for rows {start_index + 1} to {end_index}...")

for i in range(start_index, end_index):
    url = df.at[i, 'URL']
    
    # Check if URL is valid (not empty)
    if pd.notna(url) and str(url).startswith('http'):
        print(f"Scraping row {i+1}...")
        headline, body = scrape_somali_news(url)
        
        # Save the new data into the dataframe
        df.at[i, 'Headline'] = headline
        df.at[i, 'Body'] = body
        
        # Brief pause to be respectful to the server
        time.sleep(0.5)
    else:
        print(f"⚠️ Row {i+1} has no valid URL. Skipping.")

# 3. Export to a completely new file
output_file = "goobjoog_caafimaad_articles_REPAIRED.xlsx"
df.to_excel(output_file, index=False)

print(f"\n✅ Success! New data saved to: {output_file}")

🔄 Starting repair process for rows 230 to 642...
Scraping row 230...
Scraping row 231...
Scraping row 232...
Scraping row 233...


KeyboardInterrupt: 